In [1]:
import pandas as pd, numpy as np, sklearn
print("ok", pd.__version__, sklearn.__version__)

ok 2.2.3 1.6.1


In [3]:
import os
DATA = r"C:\Users\cyrin\Documents\MachineLearningCSV\MachineLearningCVE"   # remplace par TON chemin
print(sorted(os.listdir(DATA)))

['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']


In [4]:
import glob
import numpy as np, pandas as pd

SEED = 42
np.random.seed(SEED)

frames = []
for f in sorted(glob.glob(os.path.join(DATA, "*.csv"))):
    d = pd.read_csv(f, low_memory=False, encoding="latin1")
    d.columns = d.columns.str.strip()
    d["Day"] = os.path.basename(f).split("-")[0]
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
print(df.groupby("Day")["Label"].value_counts())

(2830743, 80)
Day        Label                       
Friday     BENIGN                          414322
           PortScan                        158930
           DDoS                            128027
           Bot                               1966
Monday     BENIGN                          529918
Thursday   BENIGN                          456752
           Web Attack ï¿½ Brute Force        1507
           Web Attack ï¿½ XSS                 652
           Infiltration                        36
           Web Attack ï¿½ Sql Injection        21
Tuesday    BENIGN                          432074
           FTP-Patator                       7938
           SSH-Patator                       5897
Wednesday  BENIGN                          440031
           DoS Hulk                        231073
           DoS GoldenEye                    10293
           DoS slowloris                     5796
           DoS Slowhttptest                  5499
           Heartbleed                         

In [5]:
# 1) corriger le tiret abîmé dans les noms d'attaques
df["Label"] = df["Label"].str.replace("ï¿½", "-", regex=False)

# 2) colonnes à retirer (seulement si elles existent)
drop = [c for c in ["Fwd Header Length.1", "Destination Port"] if c in df.columns]
print("colonnes retirées :", drop)
df = df.drop(columns=drop)

# 3) valeurs infinies et manquantes
num = [c for c in df.columns if c not in ("Label", "Day")]
df[num] = df[num].replace([np.inf, -np.inf], np.nan)
print("lignes avec NaN/inf :", df[num].isna().any(axis=1).sum())
df = df.dropna(subset=num)

# 4) doublons
print("doublons :", df.duplicated(subset=num).sum())
df = df.drop_duplicates(subset=num).reset_index(drop=True)

# 5) étiquette binaire : 0 = normal, 1 = attaque
df["y"] = (df["Label"] != "BENIGN").astype(int)

print(df.shape, "| part d'attaques :", round(df.y.mean(), 3))
print(df.groupby("Day")["y"].agg(["count", "sum", "mean"]))

colonnes retirées : ['Fwd Header Length.1', 'Destination Port']
lignes avec NaN/inf : 2867
doublons : 594982
(2232894, 79) | part d'attaques : 0.151
            count     sum      mean
Day                                
Friday     507190  131401  0.259076
Monday     457810       0  0.000000
Thursday   338970    2179  0.006428
Tuesday    374446    9150  0.024436
Wednesday  554478  193652  0.349251


In [6]:
import glob, os
frames = []
for f in sorted(glob.glob(os.path.join(DATA, "*.csv"))):
    d = pd.read_csv(f, low_memory=False, encoding="latin1")
    d.columns = d.columns.str.strip()
    d["Day"] = os.path.basename(f).split("-")[0]
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)

raw["Label"] = raw["Label"].str.replace("ï¿½", "-", regex=False)
raw = raw.drop(columns=[c for c in ["Fwd Header Length.1", "Destination Port"] if c in raw.columns])
num = [c for c in raw.columns if c not in ("Label", "Day")]
raw[num] = raw[num].replace([np.inf, -np.inf], np.nan)
raw = raw.dropna(subset=num)

# le lundi passe en premier : ses flux sont conservés en cas de doublon
raw = pd.concat([raw[raw.Day == "Monday"], raw[raw.Day != "Monday"]], ignore_index=True)
print("doublons :", raw.duplicated(subset=num).sum())
df = raw.drop_duplicates(subset=num).reset_index(drop=True)
df["y"] = (df["Label"] != "BENIGN").astype(int)
del raw, frames

print(df.shape)
print(df.groupby("Day")["y"].agg(["count", "sum", "mean"]))
print(df[df.y == 1].groupby(["Day", "Label"]).size())

doublons : 594982
(2232894, 79)
            count     sum      mean
Day                                
Friday     506169  131399  0.259595
Monday     458831       0  0.000000
Thursday   338970    2179  0.006428
Tuesday    374446    9150  0.024436
Wednesday  554478  193652  0.349251
Day        Label                     
Friday     Bot                             1429
           DDoS                          128014
           PortScan                        1956
Thursday   Infiltration                      36
           Web Attack - Brute Force        1470
           Web Attack - Sql Injection        21
           Web Attack - XSS                 652
Tuesday    FTP-Patator                     5931
           SSH-Patator                     3219
Wednesday  DoS GoldenEye                  10286
           DoS Hulk                      172744
           DoS Slowhttptest                5228
           DoS slowloris                   5383
           Heartbleed                        11
dtype:

In [7]:
SEED = 42

train_full = df[df.Day == "Monday"]
val  = df[df.Day.isin(["Tuesday", "Wednesday"])]
test = df[df.Day.isin(["Thursday", "Friday"])]

# l'entraînement : 30 000 flux normaux, le même volume pour tous les détecteurs
train = train_full.sample(n=30000, random_state=SEED)

assert train.y.sum() == 0, "le lundi contient des attaques !"
print("train :", len(train), "| val :", len(val), "| test :", len(test))
print("part d'attaques  val :", round(val.y.mean(), 4), "| test :", round(test.y.mean(), 4))

# vérification : aucun flux identique entre les jeux
h = lambda d: set(pd.util.hash_pandas_object(d[num], index=False))
htr, hva, hte = h(train), h(val), h(test)
print("recouvrement train/val :", len(htr & hva))
print("recouvrement train/test :", len(htr & hte))
print("recouvrement val/test :", len(hva & hte))

train : 30000 | val : 928924 | test : 845139
part d'attaques  val : 0.2183 | test : 0.1581
recouvrement train/val : 0
recouvrement train/test : 0
recouvrement val/test : 0


In [8]:
from sklearn.preprocessing import StandardScaler

# 1) garder seulement les colonnes qui varient dans l'entraînement
feats = [c for c in num if train[c].nunique() > 1]
print(len(feats), "colonnes gardées sur", len(num))

# 2) logarithme qui accepte les nombres négatifs
def slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

# 3) la normalisation apprend sur l'entraînement seulement
scaler = StandardScaler().fit(slog(train[feats].values))

# 4) on applique les mêmes réglages partout, puis on plafonne à ±10
def prep(d):
    return np.clip(scaler.transform(slog(d[feats].values)), -10, 10)

Xtr, Xva, Xte = prep(train), prep(val), prep(test)
yva, yte = val.y.values, test.y.values

print(Xtr.shape, Xva.shape, Xte.shape)

66 colonnes gardées sur 76
(30000, 66) (928924, 66) (845139, 66)


In [9]:
print("NaN :", np.isnan(Xtr).sum(), np.isnan(Xva).sum(), np.isnan(Xte).sum())

NaN : 0 0 0


In [10]:
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import average_precision_score

# le modèle apprend à quoi ressemble le trafic normal (lundi)
maha = EmpiricalCovariance().fit(Xtr)

# score d'anomalie de chaque flux de la validation : plus c'est grand, plus c'est suspect
score_maha_va = maha.mahalanobis(Xva)

# on mesure la qualité avec les étiquettes de la validation
pr_auc = average_precision_score(yva, score_maha_va)
print("PR-AUC baseline (validation) :", round(pr_auc, 4))
print("référence au hasard          :", round(yva.mean(), 4))

PR-AUC baseline (validation) : 0.7798
référence au hasard          : 0.2183


In [11]:
lab = val["Label"].values
for fam in sorted(set(lab) - {"BENIGN"}):
    m = (lab == fam) | (lab == "BENIGN")
    a = average_precision_score(yva[m], score_maha_va[m])
    print(f"{fam:20s} n={int((lab == fam).sum()):7d}  PR-AUC={a:.3f}")

DoS GoldenEye        n=  10286  PR-AUC=0.095
DoS Hulk             n= 172744  PR-AUC=0.766
DoS Slowhttptest     n=   5228  PR-AUC=0.279
DoS slowloris        n=   5383  PR-AUC=0.244
FTP-Patator          n=   5931  PR-AUC=0.021
Heartbleed           n=     11  PR-AUC=0.004
SSH-Patator          n=   3219  PR-AUC=0.009


In [12]:
from sklearn.ensemble import IsolationForest
import time

lab = val["Label"].values

def par_famille(score):
    out = {}
    for fam in sorted(set(lab) - {"BENIGN"}):
        m = (lab == fam) | (lab == "BENIGN")
        out[fam] = round(average_precision_score(yva[m], score[m]), 3)
    return out

res = []
modeles = {}
for ms in [256, 1024, 5000]:
    t0 = time.time()
    m = IsolationForest(n_estimators=200, max_samples=ms,
                        random_state=SEED, n_jobs=-1).fit(Xtr)
    s = -m.score_samples(Xva)          # plus c'est grand, plus c'est suspect
    modeles[ms] = (m, s)
    res.append((ms, round(average_precision_score(yva, s), 4), round(time.time() - t0, 1)))
    print("max_samples =", ms, "| PR-AUC =", res[-1][1], "| temps (s) =", res[-1][2])

max_samples = 256 | PR-AUC = 0.6393 | temps (s) = 18.6
max_samples = 1024 | PR-AUC = 0.6997 | temps (s) = 20.8
max_samples = 5000 | PR-AUC = 0.7665 | temps (s) = 26.9


In [13]:
for ms in [10000, 30000]:
    t0 = time.time()
    m = IsolationForest(n_estimators=200, max_samples=ms,
                        random_state=SEED, n_jobs=-1).fit(Xtr)
    s = -m.score_samples(Xva)
    modeles[ms] = (m, s)
    print("max_samples =", ms, "| PR-AUC =", round(average_precision_score(yva, s), 4),
          "| temps (s) =", round(time.time() - t0, 1))

max_samples = 10000 | PR-AUC = 0.7935 | temps (s) = 30.2
max_samples = 30000 | PR-AUC = 0.8263 | temps (s) = 60.9


In [14]:
best_ms = 30000
s_iso = modeles[best_ms][1]

fam_maha = par_famille(score_maha_va)
fam_iso = par_famille(s_iso)

cmp = pd.DataFrame({"Mahalanobis": fam_maha, "IsolationForest": fam_iso})
cmp["nb_flux"] = [int((lab == f).sum()) for f in cmp.index]
print(cmp)

                  Mahalanobis  IsolationForest  nb_flux
DoS GoldenEye           0.095            0.099    10286
DoS Hulk                0.766            0.819   172744
DoS Slowhttptest        0.279            0.200     5228
DoS slowloris           0.244            0.179     5383
FTP-Patator             0.021            0.065     5931
Heartbleed              0.004            0.038       11
SSH-Patator             0.009            0.012     3219


In [15]:
import time
from sklearn.svm import OneClassSVM

svm_modeles = {}
for nu in [0.0001, 0.0005]:
    t0 = time.time()
    m = OneClassSVM(kernel="rbf", gamma="scale", nu=nu).fit(Xtr)
    t_fit = time.time() - t0
    s = -m.decision_function(Xva)      # plus c'est grand, plus c'est suspect
    svm_modeles[nu] = (m, s)
    print("nu =", nu,
          "| PR-AUC =", round(average_precision_score(yva, s), 4),
          "| entraînement (s) =", round(t_fit, 1),
          "| total (s) =", round(time.time() - t0, 1))

nu = 0.0001 | PR-AUC = 0.8787 | entraînement (s) = 0.8 | total (s) = 33.4
nu = 0.0005 | PR-AUC = 0.8791 | entraînement (s) = 0.4 | total (s) = 18.9


In [16]:
best_nu = 0.0005
s_svm = svm_modeles[best_nu][1]

np.save("score_maha_va.npy", score_maha_va)
np.save("score_iso_va.npy", s_iso)
np.save("score_svm_va.npy", s_svm)

cmp = pd.DataFrame({"Mahalanobis": par_famille(score_maha_va),
                    "IsolationForest": par_famille(s_iso),
                    "OneClassSVM": par_famille(s_svm)})
cmp["nb_flux"] = [int((lab == f).sum()) for f in cmp.index]
print(cmp)

                  Mahalanobis  IsolationForest  OneClassSVM  nb_flux
DoS GoldenEye           0.095            0.099        0.216    10286
DoS Hulk                0.766            0.819        0.888   172744
DoS Slowhttptest        0.279            0.200        0.190     5228
DoS slowloris           0.244            0.179        0.401     5383
FTP-Patator             0.021            0.065        0.020     5931
Heartbleed              0.004            0.038        0.004       11
SSH-Patator             0.009            0.012        0.006     3219


In [17]:
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

pca = PCA(n_components=10, random_state=SEED).fit(Xtr)
Ztr, Zva = pca.transform(Xtr), pca.transform(Xva)
print("information gardée par l'ACP :", round(pca.explained_variance_ratio_.sum(), 3))

db_modeles = {}
for eps in [7, 10, 15]:
    t0 = time.time()
    db = DBSCAN(eps=eps, min_samples=10, n_jobs=-1).fit(Ztr)
    n_coeur = len(db.components_)
    n_clusters = len(set(db.labels_) - {-1})
    part_bruit = (db.labels_ == -1).mean()
    if n_coeur == 0:
        print("eps =", eps, "| aucun point cœur, on passe")
        continue
    nn = NearestNeighbors(n_neighbors=1).fit(db.components_)
    s = nn.kneighbors(Zva)[0].ravel()      # distance au point cœur le plus proche
    db_modeles[eps] = (nn, s)
    print("eps =", eps,
          "| clusters =", n_clusters,
          "| points cœurs =", n_coeur,
          "| bruit =", round(part_bruit, 3),
          "| PR-AUC =", round(average_precision_score(yva, s), 4),
          "| total (s) =", round(time.time() - t0, 1))

information gardée par l'ACP : 0.926
eps = 7 | clusters = 2 | points cœurs = 29990 | bruit = 0.0 | PR-AUC = 0.8733 | total (s) = 47.9
eps = 10 | clusters = 1 | points cœurs = 29998 | bruit = 0.0 | PR-AUC = 0.8742 | total (s) = 94.1
eps = 15 | clusters = 1 | points cœurs = 30000 | bruit = 0.0 | PR-AUC = 0.8751 | total (s) = 55.6


In [18]:
# Choix final pour DBSCAN : eps = 7 (garde 2 clusters, donc un vrai comportement de clustering ;
# PR-AUC 0,8733 contre 0,8742 pour eps = 10, écart négligeable)
best_eps = 7
s_db = db_modeles[best_eps][1]
np.save("score_db_va.npy", s_db)
print("PR-AUC DBSCAN (eps = 7) :", round(average_precision_score(yva, s_db), 4))

cmp = pd.DataFrame({
    "Mahalanobis": par_famille(score_maha_va),
    "IsolationForest": par_famille(s_iso),
    "OneClassSVM": par_famille(s_svm),
    "DBSCAN": par_famille(s_db),
})
cmp["nb_flux"] = [int((lab == f).sum()) for f in cmp.index]
print(cmp)

PR-AUC DBSCAN (eps = 7) : 0.8733
                  Mahalanobis  IsolationForest  OneClassSVM  DBSCAN  nb_flux
DoS GoldenEye           0.095            0.099        0.216   0.184    10286
DoS Hulk                0.766            0.819        0.888   0.873   172744
DoS Slowhttptest        0.279            0.200        0.190   0.112     5228
DoS slowloris           0.244            0.179        0.401   0.352     5383
FTP-Patator             0.021            0.065        0.020   0.226     5931
Heartbleed              0.004            0.038        0.004   0.003       11
SSH-Patator             0.009            0.012        0.006   0.023     3219


In [19]:
print(len(score_maha_va), len(s_iso), len(s_svm), len(s_db), len(yva))

928924 928924 928924 928924 928924


In [20]:
import json
from sklearn.metrics import precision_recall_fscore_support

dets_va = {'Mahalanobis': score_maha_va, 'IsolationForest': s_iso,
           'OneClassSVM': s_svm, 'DBSCAN': s_db}
rows, seuils = [], {}
for nom, s in dets_va.items():
    seuils[nom] = {}
    for q in [0.90, 0.95, 0.99]:
        thr = float(np.quantile(s[yva == 0], q))      # seuil appris sur les flux NORMAUX seulement
        seuils[nom][str(q)] = thr
        pred = (s > thr).astype(int)                   # 1 = alerte
        p, r, f1, _ = precision_recall_fscore_support(yva, pred, average='binary', zero_division=0)
        rows.append({'détecteur': nom, 'quantile': q,
                     'fausses_alertes_normal': round(pred[yva == 0].mean(), 3),
                     'précision': round(p, 3), 'rappel': round(r, 3), 'F1': round(f1, 3),
                     'alertes_par_10000': round(pred.mean() * 10000)})
tab_seuils = pd.DataFrame(rows)
print(tab_seuils.to_string(index=False))
json.dump(seuils, open('seuils.json', 'w'), indent=2)

      détecteur  quantile  fausses_alertes_normal  précision  rappel    F1  alertes_par_10000
    Mahalanobis      0.90                    0.10      0.703   0.849 0.769               2635
    Mahalanobis      0.95                    0.05      0.813   0.781 0.797               2095
    Mahalanobis      0.99                    0.01      0.668   0.072 0.130                236
IsolationForest      0.90                    0.10      0.722   0.932 0.814               2816
IsolationForest      0.95                    0.05      0.820   0.814 0.817               2167
IsolationForest      0.99                    0.01      0.766   0.117 0.204                335
    OneClassSVM      0.90                    0.10      0.708   0.869 0.780               2678
    OneClassSVM      0.95                    0.05      0.826   0.848 0.837               2242
    OneClassSVM      0.99                    0.01      0.918   0.403 0.560                958
         DBSCAN      0.90                    0.10      0.715

In [21]:
tab_seuils.to_csv('tableau_seuils_validation.csv', index=False)
print(json.load(open('seuils.json')))

{'Mahalanobis': {'0.9': 120.71934570521991, '0.95': 203.98155724583606, '0.99': 482.5737533006064}, 'IsolationForest': {'0.9': 0.48671491173537645, '0.95': 0.5252476349074084, '0.99': 0.6089412681974701}, 'OneClassSVM': {'0.9': -0.08047080310895934, '0.95': -0.044484500247117656, '0.99': 0.03782060243079848}, 'DBSCAN': {'0.9': 1.0436232312668385, '0.95': 1.3803683843196803, '0.99': 2.1889185508581734}}


In [22]:
print(len(score_maha_va), len(s_iso), len(s_svm), len(s_db), len(yva))

928924 928924 928924 928924 928924


In [23]:
# ÉTAPE K : TEST FINAL — aucun réglage n'est modifié après cette cellule
import json
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

seuils = json.load(open('seuils.json'))
yte = test.y.values

# Sécurité : le seuil DBSCAN enregistré correspond bien à eps = 7
assert best_eps == 7
assert np.isclose(np.quantile(s_db[yva == 0], 0.95), seuils['DBSCAN']['0.95'])

# Les modèles déjà entraînés sur le lundi (on ne ré-entraîne rien)
iso = modeles[best_ms][0]
svm = svm_modeles[best_nu][0]
nn_db = db_modeles[best_eps][0]
Zte = pca.transform(Xte)

dets_te = {'Mahalanobis': maha.mahalanobis(Xte),
           'IsolationForest': -iso.score_samples(Xte),
           'OneClassSVM': -svm.decision_function(Xte),
           'DBSCAN': nn_db.kneighbors(Zte)[0].ravel()}

rows = []
for nom, s in dets_te.items():
    np.save(f'score_{nom}_te.npy', s)
    pred = (s > seuils[nom]['0.95']).astype(int)          # seuil figé (quantile 95 %)
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average='binary', zero_division=0)
    vn, fp, fn, vp = confusion_matrix(yte, pred).ravel()
    rows.append({'détecteur': nom,
                 'PR-AUC': round(average_precision_score(yte, s), 4),
                 'précision': round(p, 3), 'rappel': round(r, 3), 'F1': round(f1, 3),
                 'VP': vp, 'FP': fp, 'FN': fn, 'VN': vn,
                 'alertes_par_10000': round(pred.mean() * 10000),
                 'fausses_alertes_normal': round(pred[yte == 0].mean(), 3)})

res_test = pd.DataFrame(rows)
print('Hasard (part d\'attaques dans le test) :', round(yte.mean(), 4))
print(res_test.to_string(index=False))
res_test.to_csv('resultats_test.csv', index=False)

Hasard (part d'attaques dans le test) : 0.1581
      détecteur  PR-AUC  précision  rappel    F1     VP    FP    FN     VN  alertes_par_10000  fausses_alertes_normal
    Mahalanobis  0.4907      0.424   0.309 0.357  41221 56024 92357 655537               1151                   0.079
IsolationForest  0.5332      0.642   0.831 0.724 111049 61936 22529 649625               2047                   0.087
    OneClassSVM  0.4604      0.613   0.612 0.612  81712 51567 51866 659994               1577                   0.072
         DBSCAN  0.5279      0.693   0.773 0.731 103296 45713 30282 665848               1763                   0.064


In [24]:
lab_te = test['Label'].values
rows_auc, rows_rap = [], []
for fam in sorted(set(lab_te) - {'BENIGN'}):
    m = (lab_te == fam) | (lab_te == 'BENIGN')      # cette famille + le trafic normal
    est_fam = lab_te == fam
    base = {'famille': fam, 'nb_flux': int(est_fam.sum())}
    auc = dict(base, hasard=round(est_fam.sum() / m.sum(), 4))
    rap = dict(base)
    for nom, s in dets_te.items():
        auc[nom] = round(average_precision_score(yte[m], s[m]), 3)
        rap[nom] = round((s[est_fam] > seuils[nom]['0.95']).mean(), 3)
    rows_auc.append(auc); rows_rap.append(rap)

fam_auc = pd.DataFrame(rows_auc).set_index('famille')
fam_rap = pd.DataFrame(rows_rap).set_index('famille')
print('PR-AUC par famille (test) :'); print(fam_auc.to_string())
print('\nRappel au seuil 95 % par famille (test) :'); print(fam_rap.to_string())
fam_auc.to_csv('test_famille_prauc.csv'); fam_rap.to_csv('test_famille_rappel.csv')

PR-AUC par famille (test) :
                            nb_flux  hasard  Mahalanobis  IsolationForest  OneClassSVM  DBSCAN
famille                                                                                       
Bot                            1429  0.0020        0.003            0.006        0.002   0.007
DDoS                         128014  0.1525        0.488            0.531        0.459   0.524
Infiltration                     36  0.0001        0.001            0.004        0.010   0.001
PortScan                       1956  0.0027        0.014            0.017        0.005   0.027
Web Attack - Brute Force       1470  0.0021        0.005            0.005        0.011   0.004
Web Attack - Sql Injection       21  0.0000        0.000            0.000        0.000   0.000
Web Attack - XSS                652  0.0009        0.003            0.002        0.005   0.002

Rappel au seuil 95 % par famille (test) :
                            nb_flux  Mahalanobis  IsolationForest  OneClas